In [1]:
import polars as pl

In [2]:
df = pl.scan_parquet('/home/ma/a/alb25/Project/thesis_code/data/raw')

In [ ]:
# time	source_user@domain	destination_user@domain	source_computer	destination_computer	authentication_type	logon_type	authentication_orientation	success/failure
df.head().collect()

time,source_user@domain,destination_user@domain,source_computer,destination_computer,authentication_type,logon_type,authentication_orientation,success/failure
str,str,str,str,str,str,str,str,str
"""1""","""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C1250""","""C586""","""NTLM""","""Network""","""LogOn""","""Success"""
"""1""","""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C586""","""C586""","""?""","""Network""","""LogOff""","""Success"""
"""1""","""C101$@DOM1""","""C101$@DOM1""","""C988""","""C988""","""?""","""Network""","""LogOff""","""Success"""
"""1""","""C1020$@DOM1""","""SYSTEM@C1020""","""C1020""","""C1020""","""Negotiate""","""Service""","""LogOn""","""Success"""
"""1""","""C1021$@DOM1""","""C1021$@DOM1""","""C1021""","""C625""","""Kerberos""","""Network""","""LogOn""","""Success"""


In [3]:
# Number of rows: 1051430459
df.select(pl.col('time').len()).collect()

time
u32
1051430459


In [3]:
source_users = df.select(pl.col('source_user@domain').unique()).collect()

In [4]:
destination_users = df.select(pl.col('destination_user@domain').unique()).collect()

In [ ]:
# 99968 users
pl.concat([source_users.select(pl.col('source_user@domain').alias('user')), destination_users.select(pl.col('destination_user@domain').alias('user'))]).unique().select(pl.len())

len
u32
99968


In [ ]:
df = df.with_columns(pl.when(pl.col('source_user@domain').str.contains(r'^U\d+@')).then(pl.lit('human'))
          .when(pl.col('source_user@domain').str.contains(r'^C\d+\$@')).then(pl.lit('machine'))
          .when(pl.col('source_user@domain').str.contains(r'^(SYSTEM|LOCAL SERVICE|NETWORK SERVICE)@')).then(pl.lit('system'))
          .when(pl.col('source_user@domain').str.contains(r'^ANONYMOUS LOGON@')).then(pl.lit('anon'))
          .otherwise(pl.lit('other'))
          .alias('source_user_type'))

In [ ]:
df.group_by('source_user_type').len().collect()